# Digital Mine IDS: SHAP experiments
**Collaboration: IBA Karachi (M. Zain Uddin, Dr. Faisal Iradat) × UEW Ghana (Digital Mine team)**

Two experiments on the **existing** Digital Mine framework (BWOA 10-feature CNN-LSTM, NSL-KDD). Nothing is retrained in Part A.

- **Part A: Explainable alerts.** SHAP on the deployed model. Which of the 10 features drive each attack class? What does an explained alert look like for an operator?
- **Part B: SHAP-based feature selection vs BWOA.** Can SHAP choose the 10 input features instead of the whale optimiser? We compare accuracy, per-class F1, stability and selection cost.

Runtime: Colab with a GPU (Runtime → Change runtime type → T4 GPU). Run all cells top to bottom. Results are written to `/content/results` and zipped at the end.

In [ ]:
# 0. Setup
!pip -q install shap xgboost
!git clone -q https://github.com/mhiskall282/Securing-the-Digital-Mine-UNESCO-Project.git mine || true
!mkdir -p mine/data/raw /content/results
!wget -q -nc https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt -O mine/data/raw/KDDTrain+.txt
!wget -q -nc https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt  -O mine/data/raw/KDDTest+.txt
!wc -l mine/data/raw/*.txt

## 1. Preprocessing (copied exactly from the team's `00_colab_setup_and_train.ipynb`)
One difference: class names come from `LabelEncoder.classes_` (alphabetical: DoS, Normal, Probe, R2L, U2R), which is the order the API uses.

In [ ]:
import os, time, json, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
os.chdir('/content/mine'); np.random.seed(42)
COL_NAMES = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot',
 'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations','num_shells',
 'num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
 'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
 'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
 'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty']
ATTACK_MAP = {'normal':'Normal',
 'neptune':'DoS','back':'DoS','land':'DoS','pod':'DoS','smurf':'DoS','teardrop':'DoS','mailbomb':'DoS','apache2':'DoS','processtable':'DoS','udpstorm':'DoS','worm':'DoS',
 'ipsweep':'Probe','nmap':'Probe','portsweep':'Probe','satan':'Probe','mscan':'Probe','saint':'Probe',
 'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L','multihop':'R2L','warezmaster':'R2L','warezclient':'R2L','spy':'R2L',
 'xlock':'R2L','xsnoop':'R2L','snmpgetattack':'R2L','snmpguess':'R2L','httptunnel':'R2L','sendmail':'R2L','named':'R2L',
 'buffer_overflow':'U2R','loadmodule':'U2R','perl':'U2R','rootkit':'U2R','sqlattack':'U2R','xterm':'U2R','ps':'U2R'}
def load(p):
    df = pd.read_csv(p, names=COL_NAMES, header=None).drop('difficulty', axis=1)
    df['label'] = df['label'].str.strip('.').map(ATTACK_MAP)
    return df.dropna(subset=['label'])
dtr, dte = load('data/raw/KDDTrain+.txt'), load('data/raw/KDDTest+.txt')
for col in ['protocol_type','service','flag']:
    le = LabelEncoder().fit(pd.concat([dtr[col], dte[col]]))
    dtr[col] = le.transform(dtr[col]); dte[col] = le.transform(dte[col])
lab = LabelEncoder().fit(['Normal','DoS','Probe','R2L','U2R'])
CLASSES = list(lab.classes_)                      # ['DoS','Normal','Probe','R2L','U2R']
y_tr, y_te = lab.transform(dtr['label']), lab.transform(dte['label'])
FEATS = COL_NAMES[:-2]
scaler = StandardScaler()
X_tr = scaler.fit_transform(dtr[FEATS].values.astype(np.float32)).astype(np.float32)
X_te = scaler.transform(dte[FEATS].values.astype(np.float32)).astype(np.float32)
print('train', X_tr.shape, 'test', X_te.shape, 'classes', CLASSES)
print('test class counts:', dict(zip(CLASSES, np.bincount(y_te))))
# sanity check against the team's saved scaler
saved = pickle.load(open('data/processed/scaler.pkl','rb'))
print('max |mean diff| vs saved scaler.pkl:', float(np.abs(saved.mean_ - scaler.mean_).max()))

In [ ]:
# 2. BWOA feature set used by the deployed model
mask = np.load('data/features/nslkdd_bwoa_mask_v3.npy'); SEL = np.where(mask == 1)[0]
BWOA_FEATS = [FEATS[i] for i in SEL]
print(len(SEL), 'BWOA features:', BWOA_FEATS)
assert BWOA_FEATS == json.load(open('data/features/nslkdd_selected_features_v3.json')), 'mask/json mismatch'
Xs_tr, Xs_te = X_tr[:, SEL], X_te[:, SEL]

## 3. Load the deployed model and reproduce the reported accuracy
The team reports **70.56%** accuracy on KDDTest+ for the BWOA v3 model. We load the Keras model (fast batch prediction for SHAP) and cross-check it against the deployed float16 TFLite model on 500 samples.

In [ ]:
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report
model = tf.keras.models.load_model('models/cnn_lstm_bwoa_v3.keras', compile=False)
def f_model(x2d):                     # (n,10) -> (n,5) probabilities
    return model.predict(np.asarray(x2d, np.float32).reshape(-1, len(SEL), 1), batch_size=8192, verbose=0)
p_te = f_model(Xs_te); yhat = p_te.argmax(1)
print(f'Accuracy {accuracy_score(y_te, yhat):.4f} | macro-F1 {f1_score(y_te, yhat, average="macro"):.4f}  (reported: 0.7056 / 0.7127)')
print(classification_report(y_te, yhat, target_names=CLASSES, digits=3, zero_division=0))
# TFLite cross-check (deployed artefact)
interp = tf.lite.Interpreter(model_path='models/cnn_lstm_quantized_float16_v3.tflite'); interp.allocate_tensors()
ii, oo = interp.get_input_details()[0]['index'], interp.get_output_details()[0]['index']
agree = 0
for x in Xs_te[:500]:
    interp.set_tensor(ii, x.reshape(1, len(SEL), 1).astype(np.float32)); interp.invoke()
    agree += int(interp.get_tensor(oo).argmax() == f_model(x[None]).argmax())
print(f'Keras vs TFLite agreement on 500 test samples: {agree/500:.1%}')

# Part A: Explainable alerts (SHAP on the deployed model)
KernelSHAP is model-agnostic. With 10 features, 1024 coalition samples is exact enumeration. Background = 50 k-means centroids of the training data. We explain a class-stratified test sample (up to 60 per class).

In [ ]:
import shap
bg = shap.kmeans(Xs_tr, 50)
idx = np.concatenate([np.random.choice(np.where(y_te == c)[0], min(60, (y_te == c).sum()), replace=False) for c in range(5)])
Xe, ye = Xs_te[idx], y_te[idx]
expl = shap.KernelExplainer(f_model, bg)
t0 = time.time(); sv = expl.shap_values(Xe, nsamples=1024, silent=True); t_shap = (time.time() - t0) / len(Xe)
sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)          # (n, 10, 5)
print('SHAP array', sv.shape, f'| {t_shap*1000:.1f} ms per explained alert (Colab)')
# additivity check: base + sum(shap) == model output
base = np.asarray(expl.expected_value); err = np.abs(base + sv.sum(1) - f_model(Xe)).max()
print('max additivity error:', float(err))

In [ ]:
# A1. Global: mean |SHAP| of the TRUE class output, per class
imp = pd.DataFrame({CLASSES[c]: np.abs(sv[ye == c, :, c]).mean(0) for c in range(5)}, index=BWOA_FEATS)
imp['overall'] = imp.mean(1); imp = imp.sort_values('overall')
imp.to_csv('/content/results/A1_shap_importance_per_class.csv')
ax = imp.drop(columns='overall').plot.barh(figsize=(8, 5), width=0.8)
ax.set_xlabel('mean |SHAP| (probability of the true class)'); ax.set_title('Which features drive each attack class (BWOA-10 CNN-LSTM)')
plt.tight_layout(); plt.savefig('/content/results/A1_importance_per_class.png', dpi=200); plt.show()
imp.round(4)

In [ ]:
# A2. Beeswarm for the DoS class (direction of effect)
c = CLASSES.index('DoS')
shap.summary_plot(sv[:, :, c], Xe, feature_names=BWOA_FEATS, show=False)
plt.title('SHAP for P(DoS)'); plt.tight_layout(); plt.savefig('/content/results/A2_beeswarm_DoS.png', dpi=200); plt.show()

In [ ]:
# A3. One "explained alert" per attack class (what an operator would see)
rows = []
pe = f_model(Xe).argmax(1)
for cname in ['DoS', 'Probe', 'R2L', 'U2R']:
    c = CLASSES.index(cname); cand = np.where((ye == c) & (pe == c))[0]
    if len(cand) == 0: print(cname, ': no correctly detected example in sample'); continue
    i = cand[0]
    e = shap.Explanation(values=sv[i, :, c], base_values=base[c], data=dte[BWOA_FEATS].values[idx[i]], feature_names=BWOA_FEATS)
    shap.plots.waterfall(e, max_display=10, show=False); plt.title(f'Explained alert: {cname}')
    plt.tight_layout(); plt.savefig(f'/content/results/A3_alert_{cname}.png', dpi=200); plt.show()
    top = np.argsort(-np.abs(sv[i, :, c]))[:3]
    rows.append({'class': cname, 'confidence': float(f_model(Xe[i:i+1])[0, c]),
                 'top_features': ', '.join(f'{BWOA_FEATS[j]} ({sv[i,j,c]:+.3f})' for j in top)})
pd.DataFrame(rows).to_csv('/content/results/A3_alert_examples.csv', index=False); pd.DataFrame(rows)

# Part B: SHAP-based feature selection vs BWOA
1. Train XGBoost on all 41 features (train set only).
2. Rank features by TreeSHAP mean |SHAP| summed over classes, and take the top 10.

**Fair comparison:** every 10-feature set is evaluated with the same classifiers, trained on KDDTrain+ and tested on KDDTest+. Test data is never used for selection.
Sets: **BWOA-10** (deployed), **SHAP-10**, **MutualInfo-10** (cheap filter baseline) and **Random-10** (5 seeds, floor).

In [ ]:
import xgboost as xgb
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
def shap_rank(seed, n=20000):
    r = np.random.RandomState(seed); s = r.choice(len(X_tr), n, replace=False)
    m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist', random_state=seed, n_jobs=-1)
    m.fit(X_tr[s], y_tr[s])
    v = shap.TreeExplainer(m).shap_values(X_tr[s[:5000]])
    v = np.stack(v, -1) if isinstance(v, list) else np.asarray(v)     # (n, 41, 5)
    return np.abs(v).mean(0).sum(-1)
t0 = time.time(); shap_imp = shap_rank(0); t_sel_shap = time.time() - t0
SHAP10 = list(np.argsort(-shap_imp)[:10])
t0 = time.time(); mi = mutual_info_classif(X_tr[:20000], y_tr[:20000], random_state=0); t_sel_mi = time.time() - t0
MI10 = list(np.argsort(-mi)[:10])
print(f'SHAP-10 ({t_sel_shap:.1f}s):', [FEATS[i] for i in SHAP10])
print(f'MI-10   ({t_sel_mi:.1f}s):', [FEATS[i] for i in MI10])
print('BWOA-10         :', BWOA_FEATS)
print('overlap SHAP & BWOA:', sorted(set(FEATS[i] for i in SHAP10) & set(BWOA_FEATS)))
pd.Series(shap_imp, index=FEATS).sort_values(ascending=False).to_csv('/content/results/B0_shap_ranking_41.csv')

In [ ]:
# B1. Stability of the SHAP top-10 across 5 resamples (Jaccard)
tops = [set(np.argsort(-shap_rank(s))[:10]) for s in range(5)]
jac = [len(a & b) / len(a | b) for i, a in enumerate(tops) for b in tops[i+1:]]
core = set.intersection(*tops)
print(f'mean pairwise Jaccard of SHAP top-10: {np.mean(jac):.3f} (min {np.min(jac):.3f})')
print('features selected in all 5 runs:', [FEATS[i] for i in sorted(core)])

In [ ]:
# B2. Evaluate every feature set with RF and XGBoost (train -> KDDTest+)
def evaluate(cols, name):
    out = []
    for clf_name, clf in [('RF', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)),
                          ('XGB', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist', random_state=0, n_jobs=-1))]:
        clf.fit(X_tr[:, cols], y_tr); p = clf.predict(X_te[:, cols]); f1c = f1_score(y_te, p, average=None, labels=range(5), zero_division=0)
        out.append({'set': name, 'clf': clf_name, 'acc': accuracy_score(y_te, p), 'macroF1': f1_score(y_te, p, average='macro'),
                    **{f'F1_{CLASSES[k]}': f1c[k] for k in range(5)}})
    return out
res = evaluate(list(SEL), 'BWOA-10') + evaluate(SHAP10, 'SHAP-10') + evaluate(MI10, 'MI-10') + evaluate(list(range(41)), 'All-41')
for s in range(5): res += evaluate(list(np.random.RandomState(100 + s).choice(41, 10, replace=False)), f'Random-10 #{s}')
R = pd.DataFrame(res); R.to_csv('/content/results/B2_feature_set_comparison.csv', index=False)
summary = R.assign(set=R['set'].str.replace(r' #\d', '', regex=True)).groupby(['set', 'clf']).mean(numeric_only=True).round(4)
summary

### B3 (optional, ~10–20 min on a T4). Retrain the team's CNN-LSTM v4 on SHAP-10 vs BWOA-10
Same architecture and hyper-parameters as the team's notebook, but with a **validation split taken from the training data**. (The test set is used only for the final evaluation.) Both sets get identical settings and seeds.

In [ ]:
import sys; sys.path.insert(0, '/content/mine')
from src.models.cnn_lstm import build_cnn_lstm_v4
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
tr_i, va_i = train_test_split(np.arange(len(X_tr)), test_size=0.15, stratify=y_tr, random_state=42)
cw = dict(enumerate(compute_class_weight('balanced', classes=np.arange(5), y=y_tr)))
def train_cnn(cols, name, epochs=30):
    tf.keras.utils.set_random_seed(42)
    m = build_cnn_lstm_v4(input_shape=(len(cols), 1), n_classes=5, filters=64, lstm_units=256, dropout_rate=0.3,
                          label_smoothing=0.1, l2_strength=1e-4, learning_rate=1e-3)
    X = lambda A: A[:, cols].reshape(-1, len(cols), 1)
    Y = lambda y: tf.keras.utils.to_categorical(y, 5)
    m.fit(X(X_tr[tr_i]), Y(y_tr[tr_i]), validation_data=(X(X_tr[va_i]), Y(y_tr[va_i])), epochs=epochs, batch_size=256,
          class_weight=cw, verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])
    p = m.predict(X(X_te), verbose=0).argmax(1); f1c = f1_score(y_te, p, average=None, labels=range(5), zero_division=0)
    return {'set': name, 'clf': 'CNN-LSTM v4', 'acc': accuracy_score(y_te, p), 'macroF1': f1_score(y_te, p, average='macro'),
            **{f'F1_{CLASSES[k]}': f1c[k] for k in range(5)}}
C = pd.DataFrame([train_cnn(list(SEL), 'BWOA-10'), train_cnn(SHAP10, 'SHAP-10')]).round(4)
C.to_csv('/content/results/B3_cnnlstm_bwoa_vs_shap.csv', index=False); C

In [ ]:
# 4. Summary + download
summ = {'reproduced_acc_BWOA10_deployed': float(accuracy_score(y_te, yhat)),
        'shap_ms_per_alert_colab': round(t_shap * 1000, 1),
        'shap_selection_seconds': round(t_sel_shap, 1),
        'shap_top10': [FEATS[i] for i in SHAP10], 'bwoa10': BWOA_FEATS,
        'shap_top10_jaccard_mean': round(float(np.mean(jac)), 3)}
json.dump(summ, open('/content/results/summary.json', 'w'), indent=2); print(json.dumps(summ, indent=2))
!cd /content && zip -qr results.zip results && ls -la results.zip
from google.colab import files; files.download('/content/results.zip')